# Scarcity-Aware Synthetic Augmentation: Kaggle GPU Execution
### Predicting When Generative Data Helps Rare Disease Classification

This notebook runs the complete end-to-end experimental benchmark on Kaggle (T4 x2 or P100):
- **Leak-free group-aware data splits** on DermaMNIST and BloodMNIST.
- **Count-matched classical controls** ($K=200$) vs. **Few-Shot DCGAN + DiffAugment**.
- **Pre-classifier diagnostic signals** (FID, KID, Diversity, Copying Distance, Confidence).
- **Downstream ResNet18 evaluation** across $N \in \{10, 20, 50, 100\}$ and 3 independent seeds.
- **Interpretable benefit prediction** (Ridge + Depth-2 Decision Tree) and zero-shot cross-modality transfer.

In [ ]:
# 1. Environment & GPU Verification
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
else:
    print("WARNING: Running on CPU. For fast execution, enable GPU T4 x2 or P100 in Session Options!")

In [ ]:
# 2. Install Required Dependencies
!pip install -q medmnist scikit-learn scipy matplotlib

In [ ]:
# 3. Generate Leak-Free Stratified Splits for Both Modalities
!python src/split.py --dataset dermamnist
!python src/split.py --dataset bloodmnist

In [ ]:
# 4. Run DermaMNIST Benchmark (39 runs: Full-data baseline + N in [10, 20, 50, 100] across 3 seeds)
!python run_experiments.py --dataset dermamnist --k 200

In [ ]:
# 5. Run BloodMNIST Cross-Modality Transfer Benchmark
!python run_experiments.py --dataset bloodmnist --k 200

In [ ]:
# 6. Generate Publication Tables & Figures (RQ1, RQ2, RQ3)
!python src/predictor.py --csv results/runs.csv

In [ ]:
# 7. Inline Visualization of Publication Figures
from IPython.display import Image, display
print("Figure 1: Downstream Rare-Class F1 vs. Scarcity N")
display(Image("results/figures/fig1_gain_vs_N.png"))
print("Figure 2: Cheap Pre-Classifier Diagnostic Signals vs. Utility Gain")
display(Image("results/figures/fig2_signals_vs_gain.png"))
print("Figure 3: Interpretable Decision Tree Rule & Cross-Modality Transfer")
display(Image("results/figures/fig3_decision_tree_rule.png"))

In [ ]:
# 8. Package All Experimental Artifacts for LaTeX Paper & Download
!zip -r results_bundle.zip results/ data/splits/ paper/ LITERATURE_REVIEW.md
print("Artifact bundle created: results_bundle.zip")